In [16]:
# pip install qdrant-client

In [17]:
import json
import time
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, asdict
from datetime import datetime
import numpy as np
 
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, 
    FieldCondition, MatchValue, HasIdCondition, Range
)

In [18]:
class QdrantVectorStore:
    """Service for storing and retrieving vectors using Qdrant."""
    
    def __init__(self, 
                 url: str = 'http://localhost:6333',
                 use_memory: bool = True):
        """
        Initialize Qdrant connection.
        
        Args:
            url: Qdrant server URL (for Docker/remote)
            use_memory: Use in-memory storage (good for testing)
        """
        
        try:
            if use_memory:
                print("Initializing Qdrant (in-memory mode)")
                self.client = QdrantClient(":memory:")
            else:
                print(f"Connecting to Qdrant at {url}")
                self.client = QdrantClient(url=url)
            
            print("✅ Connected to Qdrant successfully")
            
        except Exception as e:
            print(f"❌ Failed to connect to Qdrant: {e}")
            raise
    
    def create_collection(self, 
                         collection_name: str,
                         vector_size: int = 384,
                         distance_metric: str = 'cosine') -> bool:
        """
        Create a new collection for storing vectors.
        
        Args:
            collection_name: Name of collection
            vector_size: Dimension of vectors
            distance_metric: 'cosine', 'euclidean', or 'dot'
            
        Returns:
            True if successful
        """
        
        try:
            distance_map = {
                'cosine': Distance.COSINE,
                'euclidean': Distance.EUCLID,
                'dot': Distance.DOT
            }
            
            # Check if collection exists
            try:
                self.client.get_collection(collection_name)
                print(f"⚠️ Collection '{collection_name}' already exists")
                return True
            except:
                pass
            
            # Create collection
            self.client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(
                    size=vector_size,
                    distance=distance_map.get(distance_metric, Distance.COSINE)
                )
            )
            
            print(f"✅ Created collection: {collection_name}")
            print(f"   Vector size: {vector_size}")
            print(f"   Distance metric: {distance_metric}")
            
            return True
            
        except Exception as e:
            print(f"❌ Failed to create collection: {e}")
            return False
    
    def list_collections(self) -> List[str]:
        """Get list of all collections."""
        
        try:
            collections = self.client.get_collections()
            collection_names = [c.name for c in collections.collections]
            return collection_names
        except Exception as e:
            print(f"Error listing collections: {e}")
            return []
    
    def delete_collection(self, collection_name: str) -> bool:
        """Delete a collection."""
        
        try:
            self.client.delete_collection(collection_name)
            print(f"✅ Deleted collection: {collection_name}")
            return True
        except Exception as e:
            print(f"❌ Failed to delete collection: {e}")
            return False
 
# Test Qdrant connection
print("=" * 80)
print("TEST 1: Qdrant Setup & Connection")
print("=" * 80)
 
try:
    vector_store = QdrantVectorStore(use_memory=True)
    
    # Create test collection
    vector_store.create_collection('test_videos', vector_size=384, distance_metric='cosine')
    
    # List collections
    collections = vector_store.list_collections()
    print(f"\nAvailable collections: {collections}")
    
except Exception as e:
    print(f"Error: {e}")

TEST 1: Qdrant Setup & Connection
Initializing Qdrant (in-memory mode)
✅ Connected to Qdrant successfully
✅ Created collection: test_videos
   Vector size: 384
   Distance metric: cosine

Available collections: ['test_videos']


In [19]:
@dataclass
class VectorPayload:
    """Metadata stored with vector in Qdrant."""
    
    chunk_id: int
    video_id: str
    source: str  # 'youtube' or 'instagram'
    text: str
    timestamp_start: float = 0.0
    timestamp_end: float = 0.0
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for Qdrant payload."""
        return asdict(self)
 
def upsert_vectors_batch(vector_store: QdrantVectorStore,
                         collection_name: str,
                         vectors: List[np.ndarray],
                         payloads: List[Dict],
                         start_id: int = 1) -> bool:
    """
    Insert or update vectors in collection.
    
    Args:
        vector_store: QdrantVectorStore instance
        collection_name: Collection name
        vectors: List of embedding vectors
        payloads: List of metadata dicts
        start_id: Starting ID for vectors
        
    Returns:
        True if successful
    """
    
    try:
        points = []
        
        for idx, (vector, payload) in enumerate(zip(vectors, payloads)):
            point_id = start_id + idx
            
            point = PointStruct(
                id=point_id,
                vector=vector.tolist(),  # Convert numpy array to list
                payload=payload
            )
            points.append(point)
        
        # Upsert points (insert or update)
        vector_store.client.upsert(
            collection_name=collection_name,
            points=points
        )
        
        print(f"✅ Upserted {len(points)} vectors to '{collection_name}'")
        return True
        
    except Exception as e:
        print(f"❌ Failed to upsert vectors: {e}")
        return False
 
# Test upserting
print("\n" + "=" * 80)
print("TEST 2: Upserting Vectors with Metadata")
print("=" * 80)
 
try:
    # Create test vectors (same dimension as embeddings)
    test_vectors = [
        np.random.rand(384),
        np.random.rand(384),
        np.random.rand(384),
    ]
    
    test_payloads = [
        {
            'chunk_id': 1,
            'video_id': 'youtube_001',
            'source': 'youtube',
            'text': 'First chunk about AI and machine learning',
            'timestamp_start': 0.0,
            'timestamp_end': 30.0
        },
        {
            'chunk_id': 2,
            'video_id': 'youtube_001',
            'source': 'youtube',
            'text': 'Second chunk about deep learning networks',
            'timestamp_start': 30.0,
            'timestamp_end': 60.0
        },
        {
            'chunk_id': 1,
            'video_id': 'instagram_001',
            'source': 'instagram',
            'text': 'Instagram reel about computer vision',
            'timestamp_start': 0.0,
            'timestamp_end': 15.0
        },
    ]
    
    upsert_vectors_batch(vector_store, 'test_videos', test_vectors, test_payloads)
    
except Exception as e:
    print(f"Error: {e}")


TEST 2: Upserting Vectors with Metadata
✅ Upserted 3 vectors to 'test_videos'


In [20]:
import inspect

@dataclass
class RetrievedResult:
    """Result from similarity search."""
    
    point_id: int
    score: float
    payload: Dict
    text: str
    video_id: str
    source: str
    chunk_id: int
 
def _call_with_supported_kwargs(func, **kwargs):
    sig = inspect.signature(func)
    supported = {k: v for k, v in kwargs.items() if k in sig.parameters}
    return func(**supported)

def _extract_points(raw_results):
    if raw_results is None:
        return []
    if isinstance(raw_results, list):
        return raw_results
    if hasattr(raw_results, "points"):
        return raw_results.points
    if hasattr(raw_results, "result"):
        return raw_results.result
    return []
 
def search_similar_vectors(vector_store: QdrantVectorStore,
                          collection_name: str,
                          query_vector: np.ndarray,
                          limit: int = 5,
                          score_threshold: float = 0.5,
                          filters: Optional[Filter] = None) -> List[RetrievedResult]:
    """
    Search for similar vectors.
    
    Args:
        vector_store: QdrantVectorStore instance
        collection_name: Collection name
        query_vector: Query embedding vector
        limit: Number of results
        score_threshold: Minimum similarity score
        filters: Qdrant filter for metadata filtering
        
    Returns:
        List of RetrievedResult objects
    """
    
    try:
        client = vector_store.client
        raw_results = None
        if hasattr(client, "search"):
            raw_results = _call_with_supported_kwargs(
                client.search,
                collection_name=collection_name,
                query_vector=query_vector.tolist(),
                query_filter=filters,
                filter=filters,
                limit=limit,
                score_threshold=score_threshold
            )
        elif hasattr(client, "search_points"):
            raw_results = _call_with_supported_kwargs(
                client.search_points,
                collection_name=collection_name,
                query_vector=query_vector.tolist(),
                query_filter=filters,
                filter=filters,
                limit=limit,
                score_threshold=score_threshold
            )
        elif hasattr(client, "query_points"):
            raw_results = _call_with_supported_kwargs(
                client.query_points,
                collection_name=collection_name,
                query_vector=query_vector.tolist(),
                query=query_vector.tolist(),
                query_filter=filters,
                filter=filters,
                limit=limit,
                score_threshold=score_threshold
            )
        else:
            raise AttributeError("Qdrant client has no search method available")
        
        results = _extract_points(raw_results)
        retrieved = []
        
        for result in results:
            retrieved_result = RetrievedResult(
                point_id=result.id,
                score=result.score,
                payload=result.payload,
                text=result.payload.get('text', ''),
                video_id=result.payload.get('video_id', ''),
                source=result.payload.get('source', ''),
                chunk_id=result.payload.get('chunk_id', -1)
            )
            retrieved.append(retrieved_result)
        
        return retrieved
        
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return []
 
def search_by_query(vector_store: QdrantVectorStore,
                   collection_name: str,
                   query_embedding: np.ndarray,
                   limit: int = 5) -> List[RetrievedResult]:
    """
    Main search function for RAG.
    
    Args:
        vector_store: QdrantVectorStore instance
        collection_name: Collection name
        query_embedding: Embedding of query text
        limit: Number of results
        
    Returns:
        List of most similar chunks
    """
    
    return search_similar_vectors(
        vector_store=vector_store,
        collection_name=collection_name,
        query_vector=query_embedding,
        limit=limit,
        score_threshold=0.3  # Lower threshold for inclusion
    )
 
# Test retrieval
print("\n" + "=" * 80)
print("TEST 3: Vector Retrieval & Search")
print("=" * 80)
 
try:
    # Create a test query vector (similar to one of our test vectors)
    query_vector = test_vectors[0] + np.random.normal(0, 0.1, 384)  # Add noise
    
    # Search
    results = search_by_query(vector_store, 'test_videos', query_vector, limit=3)
    
    print(f"\nFound {len(results)} results:")
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Score: {result.score:.4f}")
        print(f"   Text: {result.text[:80]}...")
        print(f"   Video: {result.video_id} (Chunk {result.chunk_id})")
        print(f"   Source: {result.source}")
        print(f"   Timestamp: {result.payload.get('timestamp_start', 0):.1f}s - {result.payload.get('timestamp_end', 0):.1f}s")
 
except Exception as e:
    print(f"Error: {e}")


TEST 3: Vector Retrieval & Search

Found 3 results:

1. Score: 0.9869
   Text: First chunk about AI and machine learning...
   Video: youtube_001 (Chunk 1)
   Source: youtube
   Timestamp: 0.0s - 30.0s

2. Score: 0.7640
   Text: Second chunk about deep learning networks...
   Video: youtube_001 (Chunk 2)
   Source: youtube
   Timestamp: 30.0s - 60.0s

3. Score: 0.7433
   Text: Instagram reel about computer vision...
   Video: instagram_001 (Chunk 1)
   Source: instagram
   Timestamp: 0.0s - 15.0s


In [21]:
def create_video_filter(video_ids: List[str]) -> Filter:
    """Create filter for specific videos."""
    
    from qdrant_client.models import FieldCondition, MatchAny
    
    return Filter(
        must=[
            FieldCondition(
                key="video_id",
                match=MatchAny(any=video_ids)
            )
        ]
    )
 
def create_source_filter(sources: List[str]) -> Filter:
    """Create filter for video sources (youtube, instagram)."""
    
    from qdrant_client.models import FieldCondition, MatchAny
    
    return Filter(
        must=[
            FieldCondition(
                key="source",
                match=MatchAny(any=sources)
            )
        ]
    )
 
def create_combined_filter(video_ids: Optional[List[str]] = None,
                          sources: Optional[List[str]] = None) -> Optional[Filter]:
    """Create combined metadata filter."""
    
    from qdrant_client.models import FieldCondition, MatchAny
    
    conditions = []
    
    if video_ids:
        conditions.append(
            FieldCondition(
                key="video_id",
                match=MatchAny(any=video_ids)
            )
        )
    
    if sources:
        conditions.append(
            FieldCondition(
                key="source",
                match=MatchAny(any=sources)
            )
        )
    
    if not conditions:
        return None
    
    return Filter(must=conditions)
 
# Test filtering
print("\n" + "=" * 80)
print("TEST 4: Metadata Filtering")
print("=" * 80)
 
try:
    # Search only YouTube videos
    youtube_filter = create_source_filter(['youtube'])
    youtube_results = search_similar_vectors(
        vector_store, 'test_videos', query_vector, 
        limit=5, filters=youtube_filter
    )
    
    print(f"\n✅ YouTube videos found: {len(youtube_results)}")
    for result in youtube_results:
        print(f"   - {result.text[:60]}... ({result.source})")
    
    # Search specific video
    video_filter = create_video_filter(['youtube_001'])
    video_results = search_similar_vectors(
        vector_store, 'test_videos', query_vector,
        limit=5, filters=video_filter
    )
    
    print(f"\n✅ Results from youtube_001: {len(video_results)}")
    for result in video_results:
        print(f"   - Chunk {result.chunk_id}: {result.text[:60]}...")
 
except Exception as e:
    print(f"Error: {e}")


TEST 4: Metadata Filtering

✅ YouTube videos found: 2
   - First chunk about AI and machine learning... (youtube)
   - Second chunk about deep learning networks... (youtube)

✅ Results from youtube_001: 2
   - Chunk 1: First chunk about AI and machine learning...
   - Chunk 2: Second chunk about deep learning networks...


In [22]:
def get_collection_stats(vector_store: QdrantVectorStore,
                         collection_name: str) -> Dict:
    """Get statistics about a collection."""
    
    try:
        collection = vector_store.client.get_collection(collection_name)
        
        return {
            'name': collection_name,
            'vectors_count': collection.points_count,
            'vectors_size': collection.config.params.vectors.size if collection.config else None,
            'distance_metric': str(collection.config.params.vectors.distance) if collection.config else None,
            'status': str(collection.status)
        }
    except Exception as e:
        print(f"Error getting stats: {e}")
        return {}
 
def _count_value(count_result) -> int:
    if isinstance(count_result, int):
        return count_result
    if hasattr(count_result, "count"):
        return count_result.count
    if hasattr(count_result, "result"):
        result = count_result.result
        if hasattr(result, "count"):
            return result.count
        if isinstance(result, int):
            return result
    return 0
 
def delete_vectors_by_filter(vector_store: QdrantVectorStore,
                            collection_name: str,
                            video_id: str) -> int:
    """Delete all vectors for a specific video."""
    
    try:
        from qdrant_client.models import Filter, FieldCondition, MatchValue
        
        filter_condition = Filter(
            must=[
                FieldCondition(
                    key="video_id",
                    match=MatchValue(value=video_id)
                )
            ]
        )
        
        # Count before deletion
        count_before = _count_value(vector_store.client.count(collection_name))
        
        # Delete
        vector_store.client.delete(collection_name, points_selector=filter_condition)
        
        # Count after
        count_after = _count_value(vector_store.client.count(collection_name))
        deleted = max(count_before - count_after, 0)
        
        print(f"✅ Deleted {deleted} vectors for video '{video_id}'")
        return deleted
        
    except Exception as e:
        print(f"❌ Delete failed: {e}")
        return 0
 
def clear_collection(vector_store: QdrantVectorStore,
                    collection_name: str) -> bool:
    """Delete all vectors in a collection."""
    
    try:
        from qdrant_client.models import Filter, MatchAll
        
        vector_store.client.delete(
            collection_name,
            points_selector=MatchAll()
        )
        
        print(f"✅ Cleared collection: {collection_name}")
        return True
        
    except Exception as e:
        print(f"❌ Clear failed: {e}")
        return False
 
# Test management
print("\n" + "=" * 80)
print("TEST 5: Collection Management")
print("=" * 80)
 
try:
    # Get stats
    stats = get_collection_stats(vector_store, 'test_videos')
    print(f"\nCollection Stats:")
    for key, value in stats.items():
        print(f"   {key}: {value}")
    
    # Delete vectors for instagram
    deleted = delete_vectors_by_filter(vector_store, 'test_videos', 'instagram_001')
    
    # New stats
    new_stats = get_collection_stats(vector_store, 'test_videos')
    print(f"\nAfter deletion:")
    print(f"   Vectors: {new_stats.get('vectors_count', 0)}")
 
except Exception as e:
    print(f"Error: {e}")


TEST 5: Collection Management

Collection Stats:
   name: test_videos
   vectors_count: 3
   vectors_size: 384
   distance_metric: Cosine
   status: green
✅ Deleted 1 vectors for video 'instagram_001'

After deletion:
   Vectors: 2


In [23]:
class RAGVectorStore:
    """
    Production-ready interface for RAG.
    Combines all vector store operations.
    """
    
    def __init__(self, collection_name: str, vector_size: int = 384):
        """
        Initialize RAG vector store.
        
        Args:
            collection_name: Name of Qdrant collection
            vector_size: Embedding dimension
        """
        
        self.store = QdrantVectorStore(use_memory=True)
        self.collection_name = collection_name
        self.vector_size = vector_size
        
        # Create collection
        self.store.create_collection(collection_name, vector_size)
    
    def add_embedded_chunks(self, 
                           embedded_chunks: List,  # EmbeddedChunk objects
                           video_id: str) -> int:
        """
        Add embedded chunks to store.
        
        Args:
            embedded_chunks: List of EmbeddedChunk objects
            video_id: Video identifier
            
        Returns:
            Number of chunks added
        """
        
        vectors = [chunk.embedding for chunk in embedded_chunks]
        
        payloads = []
        for chunk in embedded_chunks:
            payload = {
                'chunk_id': chunk.chunk.chunk_id,
                'video_id': chunk.chunk.video_id,
                'source': chunk.chunk.source,
                'text': chunk.chunk.text,
                'timestamp_start': chunk.chunk.timestamp_start,
                'timestamp_end': chunk.chunk.timestamp_end,
            }
            payloads.append(payload)
        
        upsert_vectors_batch(
            self.store,
            self.collection_name,
            vectors,
            payloads
        )
        
        return len(vectors)
    
    def retrieve_context(self,
                        query_embedding: np.ndarray,
                        video_ids: Optional[List[str]] = None,
                        limit: int = 5) -> List[Dict]:
        """
        Retrieve relevant chunks for RAG context.
        
        Args:
            query_embedding: Query embedding
            video_ids: Optional filter by videos
            limit: Number of results
            
        Returns:
            List of relevant chunks with metadata
        """
        
        filter_cond = None
        if video_ids:
            filter_cond = create_video_filter(video_ids)
        
        results = search_similar_vectors(
            self.store,
            self.collection_name,
            query_embedding,
            limit=limit,
            filters=filter_cond
        )
        
        # Format for RAG
        formatted = []
        for result in results:
            formatted.append({
                'text': result.text,
                'score': result.score,
                'video_id': result.video_id,
                'source': result.source,
                'chunk_id': result.chunk_id,
                'timestamp': {
                    'start': result.payload.get('timestamp_start', 0),
                    'end': result.payload.get('timestamp_end', 0)
                }
            })
        
        return formatted
    
    def delete_video_data(self, video_id: str) -> bool:
        """Delete all data for a video."""
        
        delete_vectors_by_filter(self.store, self.collection_name, video_id)
        return True
    
    def get_stats(self) -> Dict:
        """Get collection statistics."""
        
        return get_collection_stats(self.store, self.collection_name)
 
# Test RAG-ready interface
print("\n" + "=" * 80)
print("TEST 6: RAG-Ready Vector Store")
print("=" * 80)
 
try:
    rag_store = RAGVectorStore(collection_name='rag_collection', vector_size=384)
    
    # Simulate adding embedded chunks
    # (In real scenario, these come from embedding pipeline)
    
    print(f"✅ RAG Vector Store initialized")
    print(f"   Collection: rag_collection")
    
    stats = rag_store.get_stats()
    print(f"   Stats: {stats}")
 
except Exception as e:
    print(f"Error: {e}")


TEST 6: RAG-Ready Vector Store
Initializing Qdrant (in-memory mode)
✅ Connected to Qdrant successfully
✅ Created collection: rag_collection
   Vector size: 384
   Distance metric: cosine
✅ RAG Vector Store initialized
   Collection: rag_collection
   Stats: {'name': 'rag_collection', 'vectors_count': 0, 'vectors_size': 384, 'distance_metric': 'Cosine', 'status': 'green'}


In [24]:
def benchmark_vector_operations(vector_store: QdrantVectorStore,
                                collection_name: str,
                                num_vectors: int = 1000):
    """
    Benchmark vector database operations.
    """
    
    print("\n" + "=" * 80)
    print(f"BENCHMARK: Vector Operations (n={num_vectors})")
    print("=" * 80)
    
    # Generate test data
    print(f"\nGenerating {num_vectors} test vectors...")
    vectors = [np.random.rand(384) for _ in range(num_vectors)]
    payloads = [
        {
            'chunk_id': i % 100,
            'video_id': f'video_{i // 100}',
            'source': 'youtube' if i % 2 == 0 else 'instagram',
            'text': f'Test chunk {i}',
            'timestamp_start': float(i),
            'timestamp_end': float(i + 10)
        }
        for i in range(num_vectors)
    ]
    
    # Benchmark: Upserting
    print("\n1. Upserting vectors...")
    start = time.time()
    upsert_vectors_batch(vector_store, collection_name, vectors, payloads)
    upsert_time = time.time() - start
    print(f"   Time: {upsert_time:.2f}s ({num_vectors/upsert_time:.0f} vectors/sec)")
    
    # Benchmark: Single search
    print("\n2. Single vector search...")
    query_vector = vectors[0]
    
    times = []
    for _ in range(10):
        start = time.time()
        results = search_similar_vectors(vector_store, collection_name, query_vector, limit=5)
        times.append(time.time() - start)
    
    avg_search_time = np.mean(times)
    print(f"   Avg time (10 searches): {avg_search_time*1000:.2f}ms")
    print(f"   Min: {np.min(times)*1000:.2f}ms, Max: {np.max(times)*1000:.2f}ms")
    
    # Benchmark: Filtered search
    print("\n3. Filtered search (by video_id)...")
    video_filter = create_video_filter(['video_0'])
    
    times = []
    for _ in range(5):
        start = time.time()
        results = search_similar_vectors(
            vector_store, collection_name, query_vector, 
            limit=5, filters=video_filter
        )
        times.append(time.time() - start)
    
    avg_filtered_time = np.mean(times)
    print(f"   Avg time: {avg_filtered_time*1000:.2f}ms")
    
    # Summary
    print("\n" + "-" * 80)
    print("SUMMARY:")
    print(f"  Upsert throughput: {num_vectors/upsert_time:.0f} vectors/sec")
    print(f"  Search latency: {avg_search_time*1000:.2f}ms")
    print(f"  Filtered search latency: {avg_filtered_time*1000:.2f}ms")
    print(f"  Overhead of filtering: {(avg_filtered_time/avg_search_time - 1)*100:.1f}%")
 
# Run benchmark (optional - takes time)
benchmark_vector_operations(vector_store, 'test_videos', num_vectors=1000)


BENCHMARK: Vector Operations (n=1000)

Generating 1000 test vectors...

1. Upserting vectors...
✅ Upserted 1000 vectors to 'test_videos'
   Time: 0.10s (10059 vectors/sec)

2. Single vector search...
   Avg time (10 searches): 0.60ms
   Min: 0.35ms, Max: 2.37ms

3. Filtered search (by video_id)...
   Avg time: 3.71ms

--------------------------------------------------------------------------------
SUMMARY:
  Upsert throughput: 10059 vectors/sec
  Search latency: 0.60ms
  Filtered search latency: 3.71ms
  Overhead of filtering: 519.9%
